In [3]:
import datetime
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split 
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score 
from sklearn.preprocessing import LabelEncoder

In [4]:
train_data = pd.read_csv("../data/fraudTrain.csv.zip")

In [5]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              

In [6]:
pd.set_option('display.max_columns', None)
train_data.head(10)

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0
5,5,2019-01-01 00:04:08,4767265376804500,"fraud_Stroman, Hudson and Erdman",gas_transport,94.63,Jennifer,Conner,F,4655 David Island,Dublin,PA,18917,40.3750,-75.2045,2158,Transport planner,1961-06-19,189a841a0a8ba03058526bcfe566aab5,1325376248,40.653382,-76.152667,0
6,6,2019-01-01 00:04:42,30074693890476,fraud_Rowe-Vandervort,grocery_net,44.54,Kelsey,Richards,F,889 Sarah Station Suite 624,Holcomb,KS,67851,37.9931,-100.9893,2691,Arboriculturist,1993-08-16,83ec1cc84142af6e2acf10c44949e720,1325376282,37.162705,-100.153370,0
7,7,2019-01-01 00:05:08,6011360759745864,fraud_Corwin-Collins,gas_transport,71.65,Steven,Williams,M,231 Flores Pass Suite 720,Edinburg,VA,22824,38.8432,-78.6003,6018,"Designer, multimedia",1947-08-21,6d294ed2cc447d2c71c7171a3d54967c,1325376308,38.948089,-78.540296,0
8,8,2019-01-01 00:05:18,4922710831011201,fraud_Herzog Ltd,misc_pos,4.27,Heather,Chase,F,6888 Hicks Stream Suite 954,Manor,PA,15665,40.3359,-79.6607,1472,Public affairs consultant,1941-03-07,fc28024ce480f8ef21a32d64c93a29f5,1325376318,40.351813,-79.958146,0
9,9,2019-01-01 00:06:01,2720830304681674,"fraud_Schoen, Kuphal and Nitzsche",grocery_pos,198.39,Melissa,Aguilar,F,21326 Taylor Squares Suite 708,Clarksville,TN,37040,36.5220,-87.3490,151785,Pathologist,1974-03-28,3b9014ea8fb80bd65de0b1463b00b00e,1325376361,37.179198,-87.485381,0


In [7]:
train_data.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud'],
      dtype='object')

In [8]:
train_data["dob"] = pd.to_datetime(train_data["dob"])

In [9]:
keep_cols = ['unix_time', 'amt', 'category', 'gender', 'dob', 
             'city_pop', 'lat', 'long', 'merch_lat', 'merch_long', 'is_fraud']
data = train_data[keep_cols].copy()

In [10]:
data

,unix_time,amt,category,gender,dob,city_pop,lat,long,merch_lat,merch_long,is_fraud
0,1325376018,4.97,misc_net,F,1988-03-09,3495,36.0788,-81.1781,36.011293,-82.048315,0
1,1325376044,107.23,grocery_pos,F,1978-06-21,149,48.8878,-118.2105,49.159047,-118.186462,0
2,1325376051,220.11,entertainment,M,1962-01-19,4154,42.1808,-112.2620,43.150704,-112.154481,0
3,1325376076,45.00,gas_transport,M,1967-01-12,1939,46.2306,-112.1138,47.034331,-112.561071,0
4,1325376186,41.96,misc_pos,M,1986-03-28,99,38.4207,-79.4629,38.674999,-78.632459,0
...,...,...,...,...,...,...,...,...,...,...,...
1296670,1371816728,15.56,entertainment,M,1961-11-24,258,37.7175,-112.4777,36.841266,-111.690765,0
1296671,1371816739,51.70,food_dining,M,1979-12-11,100,39.2667,-77.5101,38.906881,-78.246528,0
1296672,1371816752,105.93,food_dining,M,1967-08-30,899,32.9396,-105.8189,33.619513,-105.130529,0
1296673,1371816816,74.90,food_dining,M,1980-08-18,1126,43.3526,-102.5411,42.788940,-103.241160,0


# Дистанция от магазина до дома(Формула гаверсинуса)

In [11]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

In [12]:
data['distance'] = haversine_distance(
    data['lat'], data['long'],     
    data['merch_lat'], data['merch_long'] 
)

In [13]:
data['age'] = 2019 - data['dob'].dt.year

In [14]:
data

,unix_time,amt,category,gender,dob,city_pop,lat,long,merch_lat,merch_long,is_fraud,distance,age
0,1325376018,4.97,misc_net,F,1988-03-09,3495,36.0788,-81.1781,36.011293,-82.048315,0,78.597568,31
1,1325376044,107.23,grocery_pos,F,1978-06-21,149,48.8878,-118.2105,49.159047,-118.186462,0,30.212176,41
2,1325376051,220.11,entertainment,M,1962-01-19,4154,42.1808,-112.2620,43.150704,-112.154481,0,108.206083,57
3,1325376076,45.00,gas_transport,M,1967-01-12,1939,46.2306,-112.1138,47.034331,-112.561071,0,95.673231,52
4,1325376186,41.96,misc_pos,M,1986-03-28,99,38.4207,-79.4629,38.674999,-78.632459,0,77.556744,33
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,1371816728,15.56,entertainment,M,1961-11-24,258,37.7175,-112.4777,36.841266,-111.690765,0,119.752136,58
1296671,1371816739,51.70,food_dining,M,1979-12-11,100,39.2667,-77.5101,38.906881,-78.246528,0,75.104085,40
1296672,1371816752,105.93,food_dining,M,1967-08-30,899,32.9396,-105.8189,33.619513,-105.130529,0,99.047734,52
1296673,1371816816,74.90,food_dining,M,1980-08-18,1126,43.3526,-102.5411,42.788940,-103.241160,0,84.627652,39


In [15]:
data.drop(columns=['lat', 'long', 'merch_lat', 'merch_long', 'dob'], inplace=True)

In [16]:
data

,unix_time,amt,category,gender,city_pop,is_fraud,distance,age
0,1325376018,4.97,misc_net,F,3495,0,78.597568,31
1,1325376044,107.23,grocery_pos,F,149,0,30.212176,41
2,1325376051,220.11,entertainment,M,4154,0,108.206083,57
3,1325376076,45.00,gas_transport,M,1939,0,95.673231,52
4,1325376186,41.96,misc_pos,M,99,0,77.556744,33
...,...,...,...,...,...,...,...,...
1296670,1371816728,15.56,entertainment,M,258,0,119.752136,58
1296671,1371816739,51.70,food_dining,M,100,0,75.104085,40
1296672,1371816752,105.93,food_dining,M,899,0,99.047734,52
1296673,1371816816,74.90,food_dining,M,1126,0,84.627652,39


In [17]:
le_cat = LabelEncoder()
le_gen = LabelEncoder()
le_cc = LabelEncoder()

data['gender'] = le_gen.fit_transform(data['gender'])
data['category'] = le_cat.fit_transform(data['category'])

data

,unix_time,amt,category,gender,city_pop,is_fraud,distance,age
0,1325376018,4.97,8,0,3495,0,78.597568,31
1,1325376044,107.23,4,0,149,0,30.212176,41
2,1325376051,220.11,0,1,4154,0,108.206083,57
3,1325376076,45.00,2,1,1939,0,95.673231,52
4,1325376186,41.96,9,1,99,0,77.556744,33
...,...,...,...,...,...,...,...,...
1296670,1371816728,15.56,0,1,258,0,119.752136,58
1296671,1371816739,51.70,1,1,100,0,75.104085,40
1296672,1371816752,105.93,1,1,899,0,99.047734,52
1296673,1371816816,74.90,1,1,1126,0,84.627652,39


In [18]:
data['datetime'] = pd.to_datetime(data['unix_time'], unit='s')

In [19]:
data['hour'] = data['datetime'].dt.hour

In [20]:
data.drop(columns=['datetime', 'unix_time'], inplace=True)

In [21]:
data

,amt,category,gender,city_pop,is_fraud,distance,age,hour
0,4.97,8,0,3495,0,78.597568,31,0
1,107.23,4,0,149,0,30.212176,41,0
2,220.11,0,1,4154,0,108.206083,57,0
3,45.00,2,1,1939,0,95.673231,52,0
4,41.96,9,1,99,0,77.556744,33,0
...,...,...,...,...,...,...,...,...
1296670,15.56,0,1,258,0,119.752136,58,12
1296671,51.70,1,1,100,0,75.104085,40,12
1296672,105.93,1,1,899,0,99.047734,52,12
1296673,74.90,1,1,1126,0,84.627652,39,12


In [22]:
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
numeric_features = ['amt', 'city_pop', 'distance', 'age']

In [23]:
X = data.drop(columns=['is_fraud'])
y = data['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [24]:
X_train_num = scaler.fit_transform(X_train[numeric_features])
X_test_num = scaler.transform(X_test[numeric_features])

X_trainf = np.hstack([X_train_num, X_train[['category', 'gender', 'hour']].values])
X_testf = np.hstack([X_test_num, X_test[['category', 'gender', 'hour']].values])

In [25]:
from sklearn.ensemble import RandomForestClassifier

In [26]:
rf = RandomForestClassifier(class_weight='balanced', random_state=21, n_jobs=-1)
rf.fit(X_trainf, y_train)
y_pred = rf.predict(X_testf)
y_pred_proba = rf.predict_proba(X_testf)[:, 1]

In [27]:
from sklearn.metrics import average_precision_score, roc_auc_score

In [28]:
pr_auc = average_precision_score(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"PR-AUC: {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

PR-AUC: 0.9161
ROC-AUC: 0.9854


In [29]:
import xgboost as xgb

In [30]:
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=100,  
    n_estimators=500,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.8,
    random_state=21
)

In [31]:
xgb_model.fit(X_train, y_train)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
pr_auc_xgb = average_precision_score(y_test, y_proba_xgb)
roc_auc_xgb = roc_auc_score(y_test, y_proba_xgb)

print(f"XGBoost PR-AUC: {pr_auc_xgb:.4f}")
print(f"XGBoost ROC-AUC: {roc_auc_xgb:.4f}")

XGBoost PR-AUC: 0.8874
XGBoost ROC-AUC: 0.9975


In [32]:
test_data = pd.read_csv('../data/fraudTest.csv.zip')

In [33]:
test_keep_cols = ['unix_time', 'amt', 'category', 'gender', 'dob',
                   'city_pop', 'lat', 'long', 'merch_lat', 'merch_long', 'is_fraud']

In [34]:
test = test_data[test_keep_cols].copy()

In [35]:
test['distance'] = haversine_distance(
    test['lat'], test['long'],
    test['merch_lat'], test['merch_long']
)

In [36]:
test['age'] = 2019 - pd.to_datetime(test['dob']).dt.year

In [37]:
test['datetime'] = pd.to_datetime(test['unix_time'], unit='s')
test['hour'] = test['datetime'].dt.hour

In [38]:
test['gender'] = le_gen.transform(test['gender'])
test['category'] = le_cat.transform(test['category'])

In [39]:
test.drop(columns=['lat', 'long', 'merch_lat', 'merch_long', 'datetime', 'unix_time', 'dob'], inplace=True)

In [40]:
numeric_features_test = ['amt', 'city_pop', 'distance', 'age']

In [41]:
test_scaled = scaler.transform(test[numeric_features_test])

In [42]:
X_test_final = np.hstack([test_scaled, test[['category', 'gender', 'hour']].values])

In [43]:
X_test_final

array([[-0.60758977, 16.90278274, -1.24476525, ..., 10.        ,
         1.        , 12.        ],
       [-0.2406148 , -0.10998213,  0.61798099, ..., 10.        ,
         0.        , 12.        ],
       [-0.08501088,  1.63594588, -0.44465926, ...,  5.        ,
         0.        , 12.        ],
       ...,
       [ 0.53522851,  0.06270105,  0.05784285, ...,  7.        ,
         0.        , 23.        ],
       [-0.53781284, -0.11881542, -0.58713662, ..., 13.        ,
         1.        , 23.        ],
       [-0.12785637,  5.79754914, -0.13502967, ...,  0.        ,
         1.        , 23.        ]], shape=(555719, 7))

In [44]:
y_test_true = test['is_fraud']  # если нужна оценка
y_test_proba = rf.predict_proba(X_test_final)[:, 1]

In [45]:
pr_auc_test = average_precision_score(y_test_true, y_test_proba)
roc_auc_test = roc_auc_score(y_test_true, y_test_proba)

In [46]:
print(f"PR-AUC: {pr_auc_test:.4f}")
print(f"ROC-AUC: {roc_auc_test:.4f}")

PR-AUC: 0.8831
ROC-AUC: 0.9800


In [47]:
import joblib

joblib.dump(rf, '../models/fraud_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le_cat, '../models/le_cat.pkl')
joblib.dump(le_gen, '../models/le_gen.pkl')

['../models/le_gen.pkl']